In [51]:
import os
import base64
import json
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd


In [74]:
def gemma(message):
    load_dotenv()
    client = OpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    response = client.chat.completions.create(
      model="gemma-3-4b-it",
      messages=message,
      stream=True
    )

    resp = ""
    for chunk in response:
        out_token = chunk.choices[0].delta.content
        if out_token is not None: 
            resp+=out_token
            print(out_token, end="")
    print("\n")
    return resp

In [75]:
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')


In [76]:
import csv
headers = ["chunk_timestamps","llm_explanation","keywords","subtitle"]
try:
    with open("output.csv", 'w', newline='') as file:
        writer = csv.writer(file)

        # Write the headers
        writer.writerow(headers)

    print(f"output.csv created successfully with list data.")
except IOError as e:
    print(f"Error writing to output.csv: {e}")


output.csv created successfully with list data.


In [77]:
def save_seg_out(seg_out):
    seg_out = json.loads(seg_out)

    file_path = "output.csv"
    seg_df = pd.read_csv(file_path)
    seg_df = pd.concat([seg_df, pd.DataFrame(seg_out)], ignore_index=True)
    seg_df.to_csv("output.csv", index=False)

In [86]:
def msg(comb_inp_pth):
    frames_base = "/home/znyd/hacking/edu-cut/src/pre_processing/vid_frames/"
    prompt = {
           "type": "text",
           "text": """### ROLE & GOAL ###            
            ### CONTEXT & INPUT STRUCTURE ###
            I am providing you with the full content of an educational video, pre-processed and structured as a series of interleaved data chunks. You will receive the data for each 10-second segment sequentially: first, all the keyframe images for that segment, followed by a text block containing all the subtitles for that same segment. You will process all of these sequential chunks to understand the entire video.

            ### PRIMARY OBJECTIVE ###
            Your mission is to process **each 10-second segment** from the input and generate **one corresponding JSON summary object** for it.
            The most critical part of your output is the `llm_explanation` field within each object. This paragraph must be a dense, descriptive summary that synthesizes all the visual and spoken information from its corresponding 10-second clip, making the content fully understandable in isolation. This field is paramount as it will be used for vector embedding.
            Your final output must be a single, valid JSON array `[ { ... }, { ... } ]` containing one summary object for each 6 or less input segment with 10s each you were given even if they dont contain any frames.

            ### REQUIRED JSON STRUCTURE (for each object) ###
            {
            "chunk_timestamps": "string | The time range for this chunk, which you will infer from the input context (e.g., '0-10s', '10-20s'). Include every given time stamps",
            "llm_explanation": "string | A dense, self-contained descriptive paragraph explaining the key concepts, steps, and visual information presented in this chunk. This text will be used for vector embedding.",
            "keywords": "array[string] | A list of 5-7 of the most relevant keywords that summarize the content of this chunk."
            }

            ---
            ---

            ### INPUT DATA ###
            (The user will now provide the interleaved data for each 10-second segment, starting with the frames and followed by the subtitles for that segment, repeated for the entire video clip.)""" 
        }
    messages = [
    {
      "role": "user",
      "content": [prompt,
        ],
    }
    ]

    with open(comb_inp_pth, 'r', encoding='utf-8') as f:
       loaded_json = json.load(f)


    for idx, seg in enumerate(loaded_json):
        time_stamps = seg['time_stamps']
        frames = seg['frames']
        subtitle = seg['subtitle']

        messages[0]["content"].append({
              "type": "text",
              "text": f"--- DATA FOR SEGMENT {time_stamps} ---",
            })

        #Adding image frames
        if len(frames) > 0:
          for frame in frames:
              base64_image = encode_image(frames_base+frame)
              messages[0]["content"].append( {
                "type": "image_url",
                "image_url": { "url": f"data:image/png;base64,{frame}" },
              })

        #Adding subtitles for frames above
        subtitle_prompt = f"Subtitles for {time_stamps}:\n" 
        if len(subtitle) > 0:
            subtitle_prompt+="\n".join(subtitle)
            messages[0]["content"].append({
                  "type": "text",
                  "text": subtitle_prompt,
                })
        
        if (idx+1) % 6 == 0:
            print(idx) 
            if idx == 29:
              print(messages)

            # save_resp = gemma(messages)[7:][:-4]
            # save_seg_out(save_resp)
            messages[0]['content'] = [prompt,] 
        elif idx+1 == len(loaded_json):
            print(idx) 
            # save_resp = gemma(messages)[7:][:-4]
            # save_seg_out(save_resp) 
            messages[0]['content'] = [prompt,]

In [ ]:
msg("/home/znyd/hacking/edu-cut/src/pre_processing/combined_output.json")
# TODO: big problem during creating subtitle and video frame data "they are skipping"

5
11
17
23
29
[{'role': 'user', 'content': [{'type': 'text', 'text': '### ROLE & GOAL ###            \n            ### CONTEXT & INPUT STRUCTURE ###\n            I am providing you with the full content of an educational video, pre-processed and structured as a series of interleaved data chunks. You will receive the data for each 10-second segment sequentially: first, all the keyframe images for that segment, followed by a text block containing all the subtitles for that same segment. You will process all of these sequential chunks to understand the entire video.\n\n            ### PRIMARY OBJECTIVE ###\n            Your mission is to process **each 10-second segment** from the input and generate **one corresponding JSON summary object** for it.\n            The most critical part of your output is the `llm_explanation` field within each object. This paragraph must be a dense, descriptive summary that synthesizes all the visual and spoken information from its corresponding 10-second cl

In [81]:
df = pd.read_csv('output.csv')
with open("/home/znyd/hacking/edu-cut/src/pre_processing/combined_output.json", 'r', encoding='utf-8') as f:
    full_data = json.load(f)
for idx, seg in enumerate(full_data):
    sub = "\n".join(seg['subtitle'])
    df.loc[idx, 'subtitle'] = sub

df.to_csv('output.csv', index=False, quoting=csv.QUOTE_ALL)


/tmp/ipykernel_33079/188839357.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'What is the best loss function?
The age-old question in machine learning.
Will we solve this problem today?' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'subtitle'] = sub


In [ ]:
df

In [22]:
with open('output.csv', 'r', newline='') as f:
    x = csv.reader(f)
    for y in x:
        print(y)


['chunk_timestamps', 'llm_explanation', 'keywords', 'subtitle']
['0-10s', 'This segment presents the central question: ‘What is the best loss function?’ The visuals consistently display the text ‘What is the best loss function?’ across a stark white background. The video introduces the problem as an age-old challenge in machine learning and poses the question of whether it can be solved today. The repetition of the core question emphasizes its importance and sets the stage for exploring different loss functions.', "['loss function', 'machine learning', 'regression', 'optimization', 'problem']", 'What is the best loss function?\nThe age-old question in machine learning.\nWill we solve this problem today?']
['10-20s', 'The segment shifts focus to discussing loss functions, highlighting their pros and cons.  It mentions a recent paper on adaptive loss functions, suggesting a move beyond traditional approaches. The visual elements include text overlays indicating ‘Pros & Cons’ and ‘Paper D

In [83]:
with open("/home/znyd/hacking/edu-cut/src/pre_processing/combined_output.json", 'r', encoding='utf-8') as f:
    full_data = json.load(f)
print(len(full_data))

50
